# Magnitude Pruning Pipeline — Full Benchmark

**DETR-ResNet50 + YOLOv5s on COCO val2017**

Pipeline:
1. Baseline: benchmark + COCO eval
2. Global unstructured magnitude pruning at ratios 0.3, 0.5, 0.7, 0.9
3. Benchmark after pruning
4. Recovery fine-tuning (3 epochs, val2017 subset)
5. Benchmark after recovery
6. Export CSV report + plots

**Parameters benchmarked:** params, active_params, sparsity, compression_ratio, model_size_mb, sparse_size_mb, FLOPs(G), latency_ms, fps, mAP50, mAP

In [3]:
!pip install -q ultralytics
!pip install fvcore
print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires nu

In [4]:
import os, sys, copy, time, json, tempfile, warnings, zipfile
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets.utils import download_url
from torchvision.transforms import functional as TF
from PIL import Image

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

warnings.filterwarnings('ignore')
print("All imports OK.")

All imports OK.


In [5]:
# ===========================================================================
#  UNIFIED BASE MODEL INTERFACE
# ===========================================================================
class BaseModel(nn.Module):
    def forward(self, images): raise NotImplementedError
    def get_raw_model(self): raise NotImplementedError
    def get_params_count(self):
        return sum(p.numel() for p in self.get_raw_model().parameters())
    def get_input_size(self): raise NotImplementedError


def calculate_model_sparsity(model):
    total = 0; zeros = 0
    for mod in model.modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            w = mod.weight.data
            total += w.numel()
            zeros += (w == 0).sum().item()
    return zeros / max(total, 1)


# ===========================================================================
#  BASE PRUNER + MAGNITUDE PRUNER
# ===========================================================================
class BasePruner:
    def __init__(self, model, pruning_ratio=0.0):
        self.model = model
        self.pruning_ratio = pruning_ratio

    def discover_prunable_layers(self):
        return [(n, m) for n, m in self.model.named_modules()
                if isinstance(m, (nn.Conv2d, nn.Linear))]

    def register_mask(self, module, mask):
        if 'pruning_mask' in module._buffers:
            del module._buffers['pruning_mask']
        module.register_buffer('pruning_mask', mask.float())

    def apply_mask(self, module):
        if hasattr(module, 'pruning_mask'):
            module.weight.data.mul_(module.pruning_mask)

    def apply_all_masks(self):
        for _, module in self.discover_prunable_layers():
            self.apply_mask(module)

    def calculate_sparsity(self):
        t = 0; z = 0
        for _, m in self.discover_prunable_layers():
            w = m.weight.data; t += w.numel(); z += (w == 0.0).sum().item()
        return z / t if t > 0 else 0.0


class MagnitudePruner(BasePruner):
    def compute_mask(self, module):
        w = module.weight.data.abs().view(-1)
        k = int((1.0 - self.pruning_ratio) * w.numel())
        if k < 1: k = 1
        threshold = w.topk(k).values.min() if k <= w.numel() else 0.0
        return (module.weight.data.abs() >= threshold).float()

    def prune(self):
        for name, module in self.discover_prunable_layers():
            mask = self.compute_mask(module)
            self.register_mask(module, mask)
            self.apply_mask(module)


print("Base classes defined.")

Base classes defined.


In [6]:
# ===========================================================================
#  DETR MODEL WRAPPER
# ===========================================================================
class DETRModel(BaseModel):
    DETR_IMG_SIZE = 800

    def __init__(self):
        super().__init__()
        self.model = torch.hub.load('facebookresearch/detr:main', 'detr_resnet50', pretrained=True)
        self.num_classes = self.model.class_embed.out_features - 1

        def forward(self, images):
        if images.shape[-1] != self.DETR_IMG_SIZE or images.shape[-2] != self.DETR_IMG_SIZE:
            images = F.interpolate(images, size=(self.DETR_IMG_SIZE, self.DETR_IMG_SIZE),
                                   mode='bilinear', align_corners=False)
        out = self.model(images)
        logits = out['pred_logits']
        boxes = out['pred_boxes']
        prob = F.softmax(logits, dim=-1)
        scores, labels = prob[..., :-1].max(dim=-1)
        cx, cy, w, h = boxes.unbind(-1)
        x1 = (cx - w / 2).clamp(0, 1); y1 = (cy - h / 2).clamp(0, 1)
        x2 = (cx + w / 2).clamp(0, 1); y2 = (cy + h / 2).clamp(0, 1)
        boxes_xyxy = torch.stack([x1, y1, x2, y2], dim=-1)
        results = []
        for b in range(images.shape[0]):
            keep = scores[b] > 0.5
            if keep.sum() == 0:
                results.append(torch.zeros((0, 6), device=images.device))
            else:
                dets = torch.stack([
                    boxes_xyxy[b, keep, 0], boxes_xyxy[b, keep, 1],
                    boxes_xyxy[b, keep, 2], boxes_xyxy[b, keep, 3],
                    scores[b, keep], labels[b, keep].float()
                ], dim=-1)
                results.append(dets)
        return results

    def forward_train(self, images):
        B, _, H, W = images.shape
        if W != self.DETR_IMG_SIZE or H != self.DETR_IMG_SIZE:
            images = F.interpolate(images, size=(self.DETR_IMG_SIZE, self.DETR_IMG_SIZE),
                                   mode='bilinear', align_corners=False)
        out = self.model(images)
        return out['pred_logits'], out['pred_boxes']

    def get_raw_model(self): return self.model
    def get_input_size(self): return (3, self.DETR_IMG_SIZE, self.DETR_IMG_SIZE)


print("DETRModel defined.")

DETRModel defined.


In [7]:
# ===========================================================================
#  YOLOv5s MODEL WRAPPER (FIXED - FINAL)
# ===========================================================================
class YOLOv5Model(BaseModel):
    YOLO_IMG_SIZE = 640

    def __init__(self):
        super().__init__()
        from ultralytics import YOLO
        _hub = YOLO('yolov5s.pt')
        self.raw_model = _hub.model.model
        self.conf = 0.001
        self.iou = 0.65
        self.num_classes = self._infer_classes()
        print(f"YOLOv5 num_classes: {self.num_classes}")

    def _infer_classes(self):
        detect = self._get_detect()
        return detect.no - 5 if (detect is not None and hasattr(detect, 'no')) else 80

    def _get_detect(self):
        m = self.raw_model
        for i in range(len(m) - 1, -1, -1):
            if hasattr(m[i], 'no'):
                return m[i]
        return None

    def _run_backbone_neck(self, x):
        m = self.raw_model
        detect_idx = -1
        for i in range(len(m) - 1, -1, -1):
            if hasattr(m[i], 'no'):
                detect_idx = i
                break
        
        # Ensure input requires grad
        if not x.requires_grad:
            x.requires_grad_(True)
        
        layer_outputs = []
        y = x
        for i, layer in enumerate(m):
            if i == detect_idx:
                break
            
            # Set layers to train mode
            if hasattr(layer, 'train'):
                layer.train()
            
            if hasattr(layer, 'f') and layer.f != -1:
                f = layer.f
                if isinstance(f, int):
                    inp = layer_outputs[f] if f >= 0 else y
                else:
                    inp = [layer_outputs[j] if j >= 0 else y for j in f]
            else:
                inp = y
            y = layer(inp)
            layer_outputs.append(y)
        
        return [layer_outputs[j] for j in m[detect_idx].f if j >= 0]

    def _convert_to_xyxy_relative(self, pred, img_w, img_h):
        """Convert decoded predictions [cx, cy, w, h, obj_conf, cls_conf] 
        to xyxy format in relative coords [0,1]"""
        # Ensure pred has correct number of classes
        if pred.shape[-1] > 5 + self.num_classes:
            pred = pred[..., :5 + self.num_classes]
        
        box_cxcy = pred[..., 0:2]
        box_wh = pred[..., 2:4]
        x1y1 = box_cxcy - box_wh / 2.0
        x2y2 = box_cxcy + box_wh / 2.0
        boxes_xyxy = torch.cat([x1y1, x2y2], dim=-1)
        obj_conf = pred[..., 4:5]
        cls_conf = pred[..., 5:5 + self.num_classes]
        scores, labels = cls_conf.max(dim=-1, keepdim=True)
        scores = scores * obj_conf
        out = torch.cat([boxes_xyxy, scores, labels.float()], dim=-1)
        out[..., 0] /= img_w
        out[..., 1] /= img_h
        out[..., 2] /= img_w
        out[..., 3] /= img_h
        return out

    def _nms(self, pred, conf_thres=0.001, iou_thres=0.65):
        from torchvision.ops import nms
        out = []
        for i in range(pred.shape[0]):
            det = pred[i]
            mask = det[:, 4] > conf_thres
            det = det[mask]
            if det.shape[0] == 0:
                out.append(torch.zeros((0, 6), device=pred.device))
                continue
            keep = nms(det[:, :4], det[:, 4], iou_thres)
            out.append(det[keep])
        return out

    def forward(self, images):
        """Forward pass for inference/evaluation with NMS"""
        # Use forward_train to get predictions
        pred = self.forward_train(images)  # [batch, num_boxes, 5+num_classes] in pixel coords
        # Convert to relative xyxy
        pred_rel = self._convert_to_xyxy_relative(pred, self.YOLO_IMG_SIZE, self.YOLO_IMG_SIZE)
        # Apply NMS
        return self._nms(pred_rel, self.conf, self.iou)

        def forward_train(self, images):
        with torch.enable_grad():
            B, _, H, W = images.shape
            x = images * 255.0
            if W != self.YOLO_IMG_SIZE or H != self.YOLO_IMG_SIZE:
                x = F.interpolate(x, size=(self.YOLO_IMG_SIZE, self.YOLO_IMG_SIZE),
                                  mode='bilinear', align_corners=False)

            if not x.requires_grad:
                x.requires_grad_(True)

            self.raw_model.train()
            raw_out = self.raw_model(x)

            if isinstance(raw_out, dict):
                raw_out = raw_out.get('output', raw_out.get('pred', list(raw_out.values())[0]))
            elif isinstance(raw_out, (list, tuple)):
                raw_out = raw_out[0]

            if isinstance(raw_out, torch.Tensor):
                if not raw_out.requires_grad:
                    raw_out.requires_grad_(True)

                if raw_out.dim() == 3 and raw_out.shape[1] < raw_out.shape[2]:
                    raw_out = raw_out.transpose(1, 2)

                return raw_out
            else:
                raise TypeError(f"Expected torch.Tensor, got {type(raw_out)}").YOLO_IMG_SIZE)


print("YOLOv5Model defined.")

YOLOv5Model defined.


In [8]:
# ===========================================================================
#  COCO VALIDATION DATASET  (returns orig_w, orig_h for absolute pixel rescale)
# ===========================================================================
class CocoValDataset(Dataset):
    def __init__(self, root, annFile, img_size=640, max_samples=None):
        self.coco = COCO(annFile)
        self.root = root
        self.img_size = img_size
        self.ids = list(self.coco.imgs.keys())
        if max_samples is not None:
            self.ids = self.ids[:max_samples]

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.imgs[img_id]
        fpath = os.path.join(self.root, img_info['file_name'])
        img = Image.open(fpath).convert('RGB')
        orig_w, orig_h = img.size
        img = TF.resize(img, (self.img_size, self.img_size))
        return TF.to_tensor(img), img_id, orig_w, orig_h


# ===========================================================================
#  COCO TRAINING DATASET  (recovery fine-tuning)
# ===========================================================================
class CocoTrainDataset(Dataset):
    def __init__(self, root, annFile, img_size=640, offset=0, max_samples=None):
        self.coco = COCO(annFile)
        self.root = root
        self.img_size = img_size
        self.ids = list(self.coco.imgs.keys())
        if offset > 0: self.ids = self.ids[offset:]
        if max_samples is not None: self.ids = self.ids[:max_samples]

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.imgs[img_id]
        fpath = os.path.join(self.root, img_info['file_name'])
        img = Image.open(fpath).convert('RGB')
        orig_w, orig_h = img.size
        img = TF.resize(img, (self.img_size, self.img_size))
        img_tensor = TF.to_tensor(img)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        boxes = []; labels = []
        for ann in anns:
            if ann.get('ignore', 0) or ann.get('iscrowd', 0): continue
            x, y, w, h = ann['bbox']
            x1 = x / orig_w; y1 = y / orig_h
            x2 = (x + w) / orig_w; y2 = (y + h) / orig_h
            boxes.append([x1, y1, x2, y2]); labels.append(ann['category_id'] - 1)
        if not boxes:
            boxes = torch.zeros((0, 4)); labels = torch.zeros((0,), dtype=torch.long)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.long)
        return img_tensor, {'boxes': boxes, 'labels': labels}


print("Datasets defined.")

Datasets defined.


In [9]:
# ===========================================================================
#  LOSS FUNCTIONS
# ===========================================================================
def self_bbox_iou(box1, box2):
    inter_x1 = torch.max(box1[..., 0], box2[..., 0])
    inter_y1 = torch.max(box1[..., 1], box2[..., 1])
    inter_x2 = torch.min(box1[..., 2], box2[..., 2])
    inter_y2 = torch.min(box1[..., 3], box2[..., 3])
    inter = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
    area1 = (box1[..., 2] - box1[..., 0]) * (box1[..., 3] - box1[..., 1])
    area2 = (box2[..., 2] - box2[..., 0]) * (box2[..., 3] - box2[..., 1])
    return inter / (area1 + area2 - inter + 1e-9)


def compute_yolo_loss(pred, targets, num_classes=80):
    if isinstance(pred, dict):
        if 'output' in pred:
            pred = pred['output']
        elif 'pred' in pred:
            pred = pred['pred']
        else:
            pred = list(pred.values())[0]

    if isinstance(pred, (list, tuple)):
        pred = pred[0]

    if not isinstance(pred, torch.Tensor):
        raise TypeError(f"Expected torch.Tensor, got {type(pred)}")

    actual_classes = pred.shape[-1] - 5
    if actual_classes != num_classes:
        if actual_classes > 1000:
            pred = pred[..., :5 + num_classes]
            actual_classes = num_classes
        else:
            num_classes = actual_classes

    B = pred.shape[0]
    device = pred.device
    IMG_S = float(YOLOv5Model.YOLO_IMG_SIZE)

    total_loss = torch.tensor(0.0, device=device)

    for b in range(B):
        gb = targets['boxes'][b].to(device)
        gl = targets['labels'][b].to(device)

        if gb.shape[0] == 0:
            obj_t = torch.zeros(pred.shape[1], device=device)
            total_loss = total_loss + F.binary_cross_entropy_with_logits(pred[b, :, 4], obj_t, reduction='sum') * 0.1
            continue

        # gt_boxes are [x1,y1,x2,y2] in [0,1] relative
        gt_xyxy = gb * IMG_S

        # pred is [cx,cy,w,h] in pixel coords (decoded + no sigmoid on coords)
        pb_px = pred[b, :, :4]
        cx, cy = pb_px[:, 0], pb_px[:, 1]
        bw, bh = pb_px[:, 2], pb_px[:, 3]

        # Convert pred to xyxy pixel
        pb_xyxy = torch.stack([
            (cx - bw / 2).clamp(0, IMG_S),
            (cy - bh / 2).clamp(0, IMG_S),
            (cx + bw / 2).clamp(0, IMG_S),
            (cy + bh / 2).clamp(0, IMG_S),
        ], dim=-1)

        # IoU matrix
        lt = torch.max(pb_xyxy[:, None, :2], gt_xyxy[None, :, :2])
        rb = torch.min(pb_xyxy[:, None, 2:], gt_xyxy[None, :, 2:])
        wh = (rb - lt).clamp(min=0)
        inter = wh[:, :, 0] * wh[:, :, 1]
        area1 = (pb_xyxy[:, 2] - pb_xyxy[:, 0]).clamp(min=0) * (pb_xyxy[:, 3] - pb_xyxy[:, 1]).clamp(min=0)
        area2 = (gt_xyxy[:, 2] - gt_xyxy[:, 0]).clamp(min=0) * (gt_xyxy[:, 3] - gt_xyxy[:, 1]).clamp(min=0)
        iou = inter / (area1[:, None] + area2[None, :] - inter + 1e-9)

        matched_iou, matched_gt_idx = iou.max(dim=1)
        pos_mask = matched_iou > 0.3
        if pos_mask.sum() == 0:
            fb = matched_iou.argmax()
            pos_mask = torch.zeros(pred.shape[1], dtype=torch.bool, device=device)
            pos_mask[fb] = True

        # Box loss: L1 on cxcywh
        matched_cx = (gt_xyxy[matched_gt_idx, 0] + gt_xyxy[matched_gt_idx, 2]) / 2
        matched_cy = (gt_xyxy[matched_gt_idx, 1] + gt_xyxy[matched_gt_idx, 3]) / 2
        matched_w = (gt_xyxy[matched_gt_idx, 2] - gt_xyxy[matched_gt_idx, 0]).clamp(min=0)
        matched_h = (gt_xyxy[matched_gt_idx, 3] - gt_xyxy[matched_gt_idx, 1]).clamp(min=0)
        matched_cxcywh = torch.stack([matched_cx, matched_cy, matched_w, matched_h], dim=-1)

        box_loss = F.l1_loss(pb_px[pos_mask], matched_cxcywh[pos_mask], reduction='sum')

        # Class loss
        cls_logits = pred[b, pos_mask, 5:5 + num_classes]
        cls_target = F.one_hot(gl[matched_gt_idx[pos_mask]], num_classes).float()
        cls_loss = F.binary_cross_entropy_with_logits(cls_logits, cls_target, reduction='sum')

        # Objectness loss
        obj_target = torch.zeros(pred.shape[1], device=device)
        obj_target[pos_mask] = matched_iou[pos_mask].detach()
        obj_loss = F.binary_cross_entropy_with_logits(pred[b, :, 4], obj_target, reduction='sum')

        total_loss = total_loss + box_loss * 0.05 + cls_loss * 0.5 + obj_loss * 1.0

    return total_loss / max(B, 1)
    
def box_cxcywh_to_xyxy(x):
    cx, cy, w, h = x.unbind(-1)
    return torch.stack([(cx - w/2).clamp(0, 1), (cy - h/2).clamp(0, 1),
                        (cx + w/2).clamp(0, 1), (cy + h/2).clamp(0, 1)], dim=-1)


def giou_loss(boxes1, boxes2):
    x1 = torch.min(boxes1[..., 0], boxes2[..., 0])
    y1 = torch.min(boxes1[..., 1], boxes2[..., 1])
    x2 = torch.max(boxes1[..., 2], boxes2[..., 2])
    y2 = torch.max(boxes1[..., 3], boxes2[..., 3])
    inter_x1 = torch.max(boxes1[..., 0], boxes2[..., 0])
    inter_y1 = torch.max(boxes1[..., 1], boxes2[..., 1])
    inter_x2 = torch.min(boxes1[..., 2], boxes2[..., 2])
    inter_y2 = torch.min(boxes1[..., 3], boxes2[..., 3])
    inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    area1 = (boxes1[..., 2] - boxes1[..., 0]).clamp(min=0) * (boxes1[..., 3] - boxes1[..., 1]).clamp(min=0)
    area2 = (boxes2[..., 2] - boxes2[..., 0]).clamp(min=0) * (boxes2[..., 3] - boxes2[..., 1]).clamp(min=0)
    union = area1 + area2 - inter
    iou = inter / (union + 1e-9)
    enclose = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    return iou - (enclose - union) / (enclose + 1e-9)


def compute_detr_loss(pred_logits, pred_boxes, targets, num_classes=80):
    device = pred_logits.device
    B, N = pred_logits.shape[:2]
    total_loss = 0.0

    for b in range(B):
        gt_boxes = targets['boxes'][b].to(device)
        gl = targets['labels'][b].to(device)

        if gt_boxes.shape[0] == 0:
            bg = torch.full((N,), num_classes, dtype=torch.long, device=device)
            total_loss = total_loss + F.cross_entropy(pred_logits[b], bg) * 1.0
            continue

        # Convert GT from [x1,y1,x2,y2] to cxcywh for matching
        gt_cx = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
        gt_cy = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2
        gt_w = (gt_boxes[:, 2] - gt_boxes[:, 0]).clamp(min=0)
        gt_h = (gt_boxes[:, 3] - gt_boxes[:, 1]).clamp(min=0)
        gt_cxcywh = torch.stack([gt_cx, gt_cy, gt_w, gt_h], dim=-1)

        n_gt = gt_boxes.shape[0]
        pl = pred_logits[b]
        pb = pred_boxes[b]
        p_xyxy = box_cxcywh_to_xyxy(pb)
        g_xyxy = gt_boxes

        # L1 cost
        l1_cost = torch.cdist(pb, gt_cxcywh, p=1)

        # GIoU cost
        p_exp = p_xyxy[:, None, :].expand(N, n_gt, 4)
        g_exp = g_xyxy[None, :, :].expand(N, n_gt, 4)
        with torch.no_grad():
            giou = giou_loss(p_exp.reshape(-1, 4), g_exp.reshape(-1, 4)).reshape(N, n_gt)
            giou_cost = 1.0 - giou

        # Combined cost
        cost = 2.0 * l1_cost + 2.0 * giou_cost

        # Hungarian matching
        from scipy.optimize import linear_sum_assignment
        cost_np = cost.detach().cpu().numpy()
        pred_idx, gt_idx = linear_sum_assignment(cost_np)

        # CE loss: all queries, unmatched -> num_classes
        target_classes = torch.full((N,), num_classes, dtype=torch.long, device=device)
        target_classes[pred_idx] = gl[gt_idx]
        loss_ce = F.cross_entropy(pl, target_classes, reduction='sum')

        # L1 loss on matched boxes
        loss_bbox = F.l1_loss(pb[pred_idx], gt_cxcywh[gt_idx], reduction='sum')

        # GIoU loss on matched boxes
        matched_p = p_xyxy[pred_idx]
        matched_g = g_xyxy[gt_idx]
        loss_giou = (1.0 - giou_loss(matched_p, matched_g)).sum()

        total_loss = total_loss + loss_ce * 1.0 + loss_bbox * 5.0 + loss_giou * 2.0

    return total_loss / B


# Smoke test
if __name__ == '__main__':
    print("DETR loss smoke test...")
    B, N = 2, 100
    pl = torch.randn(B, N, 81, requires_grad=True)
    pb = torch.rand(B, N, 4)
    targets = {
        'boxes': [torch.rand(3, 4) * 0.8 + 0.1 for _ in range(B)],
        'labels': [torch.randint(0, 80, (3,)) for _ in range(B)]
    }
    loss = compute_detr_loss(pl, pb, targets, num_classes=80)
    print(f"  loss={loss.item():.4f}, requires_grad={loss.requires_grad}, finite={torch.isfinite(loss).item()}")
    loss.backward()
    print(f"  grad ok: {pl.grad is not None}")

Loss functions defined.


In [10]:
# ===========================================================================
#  SPARSITY ENFORCEMENT + RECOVERY TRAINING
# ===========================================================================
def enforce_sparsity(model):
    for mod in model.modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)) and hasattr(mod, 'pruning_mask'):
            mod.weight.data.mul_(mod.pruning_mask)

def recover_model(model, pruner, train_loader, device, epochs=3, lr=1e-4,
                  report_every=50, loss_fn=None, num_classes=80, model_type='yolo'):
    # Set model to train mode
    model.train()
    
    # Ensure all parameters require grad
    for param in model.parameters():
        param.requires_grad = True
    
    pruner.apply_all_masks()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        running_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f'Epoch {epoch+1}/{epochs}')
        for step, batch in enumerate(train_loader):
            # Unpack batch
            if isinstance(batch, (list, tuple)):
                images = batch[0]
                targets = batch[1]
            else:
                raise TypeError(f"Unexpected batch type: {type(batch)}")
            
            images = images.to(device)
            
            # ============================================================
            # FIX: Handle targets properly - list of dicts case
            # ============================================================
            if isinstance(targets, list):
                # Case: targets is list of dicts, each with 'boxes' and 'labels'
                if len(targets) > 0 and isinstance(targets[0], dict):
                    boxes = []
                    labels = []
                    for item in targets:
                        if 'boxes' in item and 'labels' in item:
                            boxes.append(item['boxes'].to(device))
                            labels.append(item['labels'].to(device))
                        else:
                            raise ValueError(f"Missing 'boxes' or 'labels' in dict: {item.keys()}")
                    targets = {'boxes': boxes, 'labels': labels}
                else:
                    # List of (boxes, labels) pairs
                    boxes = []
                    labels = []
                    for item in targets:
                        if isinstance(item, (list, tuple)) and len(item) == 2:
                            boxes.append(item[0].to(device))
                            labels.append(item[1].to(device))
                        else:
                            raise ValueError(f"Unexpected item in targets: {type(item)}")
                    targets = {'boxes': boxes, 'labels': labels}
            
            elif isinstance(targets, dict):
                # Case: targets is dict with 'boxes' and 'labels'
                if 'boxes' in targets and 'labels' in targets:
                    if isinstance(targets['boxes'], list):
                        targets = {
                            'boxes': [b.to(device) for b in targets['boxes']],
                            'labels': [l.to(device) for l in targets['labels']]
                        }
                    else:
                        targets = {
                            'boxes': targets['boxes'].to(device),
                            'labels': targets['labels'].to(device)
                        }
                else:
                    targets = {
                        k: v.to(device) if isinstance(v, torch.Tensor) else v
                        for k, v in targets.items()
                    }
            else:
                raise TypeError(f"Unexpected targets type: {type(targets)}")
            
            # ============================================================
            # Training step
            # ============================================================
            optimizer.zero_grad()
            pred = model.forward_train(images)
            
            # ============================================================
            # FIX: Handle pred shape for YOLO
            # ============================================================
            if model_type == 'yolo' and isinstance(pred, torch.Tensor):
                # Get actual number of classes from pred
                actual_classes = pred.shape[-1] - 5
                
                # If pred has wrong number of classes (e.g., 8395 instead of 80)
                if actual_classes != num_classes:
                    if actual_classes > 1000:
                        # Too many classes - likely incorrectly reshaped
                        # Try to reshape to correct format
                        print(f"Warning: Pred has {actual_classes} classes, slicing to {num_classes}")
                        pred = pred[..., :5 + num_classes]
                    else:
                        # Update num_classes to match pred
                        print(f"Warning: num_classes mismatch. Pred has {actual_classes}, expected {num_classes}")
                        num_classes = actual_classes
            
            # Ensure pred requires grad
            if isinstance(pred, torch.Tensor) and not pred.requires_grad:
                pred.requires_grad_(True)
            
            # Calculate loss
            if model_type == 'yolo':
                if loss_fn is not None:
                    loss = loss_fn(pred, targets, num_classes=num_classes)
                else:
                    loss = compute_yolo_loss(pred, targets, num_classes=num_classes)
            else:
                if loss_fn is not None:
                    loss = loss_fn(pred, targets, num_classes=num_classes)
                else:
                    loss = compute_detr_loss(pred[0], pred[1], targets, num_classes=num_classes)
            
            loss.backward()
            enforce_sparsity(model)
            optimizer.step()
            enforce_sparsity(model)
            running_loss += loss.item()
            
            if step % report_every == 0 and step > 0:
                pbar.set_postfix({'loss': f'{running_loss/(step+1):.4f}'})
    
    return running_loss / max(len(train_loader), 1)

In [11]:
# ===========================================================================
#  BENCHMARKING HELPERS
# ===========================================================================
def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix='.pth', delete=False) as f:
        torch.save(model.get_raw_model().state_dict(), f.name)
        sz = os.path.getsize(f.name) / (1024 * 1024)
    os.unlink(f.name)
    return sz


def theoretical_sparse_size_mb(model):
    total_bytes = 0
    for p in model.get_raw_model().parameters():
        total_bytes += (p.data != 0).sum().item() * p.element_size()
    return total_bytes / (1024 * 1024)


def benchmark_model(model, dataloader, device, desc='', num_batches=200):
    model.eval(); model.to(device)
    latencies, total_frames, total_time = [], 0, 0.0
    is_cuda = device.type == 'cuda'
    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc=desc,
                                       total=min(num_batches, len(dataloader)))):
            if i >= num_batches: break
            images = batch[0].to(device)
            if images.dim() == 3: images = images.unsqueeze(0)
            if is_cuda: torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(images)
            if is_cuda: torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0
            latencies.append(elapsed)
            total_frames += images.size(0); total_time += elapsed
    avg_lat = sum(latencies) / len(latencies) if latencies else 0
    return {'latency_ms': avg_lat * 1000, 'fps': total_frames / max(total_time, 1e-9)}


def compute_flops(model, device='cpu'):
    """Compute FLOPs for the model"""
    try:
        from fvcore.nn import FlopCountAnalysis
        import warnings
        warnings.filterwarnings('ignore', category=UserWarning)
        
        # Get raw model and move to device
        raw_model = model.get_raw_model().to(device).eval()
        
        # Create dummy input
        input_size = (1,) + model.get_input_size()
        dummy = torch.randn(*input_size, device=device)
        
        # For YOLO, we need to handle the forward pass specially
        if isinstance(model, YOLOv5Model):
            # Use the model's forward method directly
            with torch.no_grad():
                flops = FlopCountAnalysis(raw_model, dummy)
            return flops.total() / 1e9
        else:
            # For DETR and others
            with torch.no_grad():
                flops = FlopCountAnalysis(raw_model, dummy)
            return flops.total() / 1e9
    except Exception as e:
        print(f"  [FLOPs] skipped: {e}")
        return float('nan')

def evaluate_coco(model, dataloader, coco_gt, device, desc='eval'):
    model.eval(); model.to(device)
    results = []
    with torch.no_grad():
        for images, img_ids, orig_ws, orig_hs in tqdm(dataloader, desc=desc):
            images = images.to(device)
            if images.dim() == 3: images = images.unsqueeze(0)
            preds_per_image = model(images)
            for batch_idx, dets in enumerate(preds_per_image):
                img_id = (img_ids[batch_idx].item()
                          if isinstance(img_ids, torch.Tensor) else img_ids[batch_idx])
                orig_w = (orig_ws[batch_idx].item()
                          if isinstance(orig_ws, torch.Tensor) else orig_ws[batch_idx])
                orig_h = (orig_hs[batch_idx].item()
                          if isinstance(orig_hs, torch.Tensor) else orig_hs[batch_idx])
                if dets.shape[0] == 0: continue
                dets = dets.cpu().numpy()
                for d in dets:
                    x1, y1, x2, y2, score, cls_id = d
                    results.append({
                        'image_id': int(img_id),
                        'category_id': int(cls_id) + 1,
                        'bbox': [float(x1 * orig_w), float(y1 * orig_h),
                                 float((x2 - x1) * orig_w), float((y2 - y1) * orig_h)],
                        'score': float(score)
                    })
    if not results:
        print(f"  [WARN] No detections submitted for {desc}!")
        return {'mAP50': 0.0, 'mAP': 0.0, 'precision': 0.0, 'recall': 0.0}
    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()
    return {
        'mAP50': float(coco_eval.stats[1]),
        'mAP': float(coco_eval.stats[0]),
        'precision': float(coco_eval.stats[4]),
        'recall': float(coco_eval.stats[5])
    }


print("Benchmarking helpers defined.")

Benchmarking helpers defined.


In [12]:
# ===========================================================================
#  COCO VAL2017 DOWNLOAD
# ===========================================================================
DATA_DIR = "/kaggle/working/coco"
os.makedirs(DATA_DIR, exist_ok=True)

VAL_ROOT = os.path.join(DATA_DIR, "val2017")
VAL_ANN  = os.path.join(DATA_DIR, "annotations", "instances_val2017.json")

if not os.path.isdir(VAL_ROOT):
    print("Downloading COCO val2017 images...")
    download_url("http://images.cocodataset.org/zips/val2017.zip",
                 DATA_DIR, "val2017.zip")
    with zipfile.ZipFile(os.path.join(DATA_DIR, "val2017.zip"), "r") as zf:
        zf.extractall(DATA_DIR)
    os.remove(os.path.join(DATA_DIR, "val2017.zip"))

if not os.path.isfile(VAL_ANN):
    print("Downloading COCO annotations...")
    download_url(
        "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        DATA_DIR, "annotations.zip")
    with zipfile.ZipFile(os.path.join(DATA_DIR, "annotations.zip"), "r") as zf:
        zf.extractall(DATA_DIR)
    os.remove(os.path.join(DATA_DIR, "annotations.zip"))

print(f"COCO val2017 ready: {VAL_ROOT} | {VAL_ANN}")

100%|██████████| 816M/816M [00:08<00:00, 100MB/s]  


100%|██████████| 253M/253M [00:02<00:00, 101MB/s]  


COCO val2017 ready: /kaggle/working/coco/val2017 | /kaggle/working/coco/annotations/instances_val2017.json


In [13]:
# ===========================================================================
#  LOAD MODELS
# ===========================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("Loading DETR-ResNet50...")
detr = DETRModel().to(device).eval()
print(f"  DETR params: {detr.get_params_count():,}")

print("Loading YOLOv5s...")
yolo = YOLOv5Model().to(device).eval()
print(f"  YOLOv5s params: {yolo.get_params_count():,}")

print(f"\nDETR   initial sparsity: {calculate_model_sparsity(detr):.4f}")
print(f"YOLOv5 initial sparsity: {calculate_model_sparsity(yolo):.4f}")

Device: cuda
Loading DETR-ResNet50...
Downloading: "https://github.com/facebookresearch/detr/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 213MB/s]


Downloading: "https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth" to /root/.cache/torch/hub/checkpoints/detr-r50-e632da11.pth


100%|██████████| 159M/159M [00:00<00:00, 199MB/s]  


  DETR params: 41,524,768
Loading YOLOv5s...
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PRO TIP 💡 Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
YOLOv5 num_classes: 139
  YOLOv5s params: 9,153,152

DETR   initial sparsity: 0.0000
YOLOv5 initial sparsity: 0.0000


In [14]:
# ===========================================================================
#  VALIDATION DATALOADERS — separate per model (FIXED: no double-resize)
# ===========================================================================
VAL_MAX_SAMPLES = 500

val_ds_640 = CocoValDataset(VAL_ROOT, VAL_ANN, img_size=640, max_samples=VAL_MAX_SAMPLES)
val_ds_800 = CocoValDataset(VAL_ROOT, VAL_ANN, img_size=800, max_samples=VAL_MAX_SAMPLES)

val_loader_640 = DataLoader(val_ds_640, batch_size=4, shuffle=False, num_workers=2)
val_loader_800 = DataLoader(val_ds_800, batch_size=2, shuffle=False, num_workers=2)

coco_gt = COCO(VAL_ANN)
print(f"Val samples: {len(val_ds_640)}")
print(f"YOLO loader: batch=4, img=640x640")
print(f"DETR loader: batch=2, img=800x800")

loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
loading annotations into memory...
Done (t=0.73s)
creating index...
index created!
loading annotations into memory...
Done (t=0.50s)
creating index...
index created!
Val samples: 500
YOLO loader: batch=4, img=640x640
DETR loader: batch=2, img=800x800


In [15]:
# ===========================================================================
#  MAIN EXPERIMENT LOOP
# ===========================================================================
PRUNING_RATIOS = [0.0, 0.3, 0.5, 0.7, 0.9]
REPORT = []

RECOVERY_OFFSET = VAL_MAX_SAMPLES
RECOVERY_SAMPLES = 200

MODEL_CONFIGS = [
    ("YOLOv5", yolo, val_loader_640),
    ("DETR",   detr, val_loader_800),
    
]

for model_name, model_obj, val_loader in MODEL_CONFIGS:
    print(f"\n{'='*65}")
    print(f"MODEL: {model_name}  |  input: {model_obj.get_input_size()}")
    print(f"{'='*65}")

    baseline_params = model_obj.get_params_count()

    for ratio in PRUNING_RATIOS:
        print(f"\n--- ratio={ratio} ---")

        if ratio == 0.0:
            row = {"model": model_name, "pruning_ratio": ratio, "stage": "baseline"}
            sp = calculate_model_sparsity(model_obj.get_raw_model())
            row["params"] = baseline_params
            row["active_params"] = baseline_params
            row["sparsity"] = sp
            row["compression_ratio"] = 1.0
            row["model_size_mb"] = model_size_mb(model_obj)
            row["sparse_size_mb"] = theoretical_sparse_size_mb(model_obj)
            row["flops_g"] = compute_flops(model_obj, device=str(device))

            bench = benchmark_model(model_obj, val_loader, device,
                                    desc=f"{model_name} baseline bench", num_batches=200)
            row.update(bench)
            metrics = evaluate_coco(model_obj, val_loader, coco_gt, device,
                                     desc=f"{model_name} baseline eval")
            row.update(metrics)
            REPORT.append(row)
            print(f"  Baseline: mAP={metrics['mAP']:.3f}, mAP50={metrics['mAP50']:.3f}, "
                  f"FPS={row['fps']:.1f}, Sparsity={sp*100:.1f}%")
        else:
            cloned = copy.deepcopy(model_obj)
            raw_cloned = cloned.get_raw_model()
            pruner = MagnitudePruner(raw_cloned, pruning_ratio=ratio)
            pruner.prune()
            actual_sparsity = pruner.calculate_sparsity()

            row = {"model": model_name, "pruning_ratio": ratio, "stage": "after_pruning"}
            row["params"] = cloned.get_params_count()
            row["active_params"] = int(row["params"] * (1 - actual_sparsity))
            row["sparsity"] = actual_sparsity
            row["compression_ratio"] = 1.0 / (1.0 - actual_sparsity + 1e-9)
            row["model_size_mb"] = model_size_mb(cloned)
            row["sparse_size_mb"] = theoretical_sparse_size_mb(cloned)
            row["flops_g"] = compute_flops(cloned, device=str(device))

            bench = benchmark_model(cloned, val_loader, device,
                                    desc=f"{model_name} pruned {ratio}", num_batches=200)
            row.update(bench)
            metrics = evaluate_coco(cloned, val_loader, coco_gt, device,
                                     desc=f"{model_name} pruned {ratio} eval")
            row.update(metrics)
            REPORT.append(row)
            print(f"  Pruned ({ratio:.0%}): mAP={metrics['mAP']:.3f}, mAP50={metrics['mAP50']:.3f}, "
                  f"FPS={row['fps']:.1f}, Sparsity={actual_sparsity*100:.1f}%")

            # Recovery fine-tuning
            print(f"  Recovery training on {RECOVERY_SAMPLES} samples...")
            train_loader = DataLoader(
                CocoTrainDataset(
                    VAL_ROOT, VAL_ANN,
                    img_size=model_obj.YOLO_IMG_SIZE if model_name == "YOLOv5" else model_obj.DETR_IMG_SIZE,
                    offset=RECOVERY_OFFSET, max_samples=RECOVERY_SAMPLES
                ), batch_size=2, shuffle=True, num_workers=2,
                collate_fn=lambda batch: (
                    torch.stack([b[0] for b in batch]),
                    [{k: v for k, v in b[1].items()} for b in batch]
                ))

            if model_name == "YOLOv5":
                _ = recover_model(cloned, pruner, train_loader, device,
                                  epochs=3, lr=1e-4, num_classes=cloned.num_classes,
                                  model_type='yolo')
            else:
                _ = recover_model(cloned, pruner, train_loader, device,
                                  epochs=3, lr=1e-4, num_classes=cloned.num_classes,
                                  model_type='detr',
                                  loss_fn=lambda p, t, nc: compute_detr_loss(*p, t, num_classes=nc))

            # after_recovery
            row = {"model": model_name, "pruning_ratio": ratio, "stage": "after_recovery"}
            row["params"] = cloned.get_params_count()
            row["active_params"] = int(row["params"] * (1 - actual_sparsity))
            row["sparsity"] = actual_sparsity
            row["compression_ratio"] = 1.0 / (1.0 - actual_sparsity + 1e-9)
            row["model_size_mb"] = model_size_mb(cloned)
            row["sparse_size_mb"] = theoretical_sparse_size_mb(cloned)
            row["flops_g"] = compute_flops(cloned, device=str(device))

            bench = benchmark_model(cloned, val_loader, device,
                                    desc=f"{model_name} recovered {ratio}", num_batches=200)
            row.update(bench)
            metrics = evaluate_coco(cloned, val_loader, coco_gt, device,
                                     desc=f"{model_name} recovered {ratio} eval")
            row.update(metrics)
            REPORT.append(row)
            print(f"  Recovered ({ratio:.0%}): mAP={metrics['mAP']:.3f}, "
                  f"mAP50={metrics['mAP50']:.3f}, FPS={row['fps']:.1f}")


print(f"\n{'='*65}")
print("EXPERIMENT COMPLETE")
print(f"{'='*65}")


MODEL: YOLOv5  |  input: (3, 640, 640)

--- ratio=0.0 ---
  [FLOPs] skipped: cat() received an invalid combination of arguments - got (Tensor, int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)



YOLOv5 baseline bench:   0%|          | 0/125 [00:00<?, ?it/s]

YOLOv5 baseline eval:   0%|          | 0/125 [00:00<?, ?it/s]

  [DEBUG] YOLOv5 baseline eval: 22394 detections submitted
Loading and preparing results...
DONE (t=0.44s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.18s).
Accumulating evaluation results...
DONE (t=1.44s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average 

YOLOv5 pruned 0.3:   0%|          | 0/125 [00:00<?, ?it/s]

...

YOLOv5 recovered 0.3 eval:   0%|          | 0/125 [00:00<?, ?it/s]

  [DEBUG] YOLOv5 recovered 0.3 eval: 6830 detections submitted
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.93s).
Accumulating evaluation results...
DONE (t=1.26s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Aver

YOLOv5 pruned 0.5:   0%|          | 0/125 [00:00<?, ?it/s]

YOLOv5 pruned 0.5 eval:   0%|          | 0/125 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ===========================================================================
#  RESULTS TABLE
# ===========================================================================
df = pd.DataFrame(REPORT)
stage_order = {"baseline": 0, "after_pruning": 1, "after_recovery": 2}
df["stage_order"] = df["stage"].map(stage_order)
df = df.sort_values(["model", "stage_order", "pruning_ratio"]).drop(columns="stage_order")

display_cols = ["model", "stage", "pruning_ratio", "params", "active_params",
                "sparsity", "compression_ratio", "model_size_mb", "sparse_size_mb",
                "flops_g", "latency_ms", "fps", "mAP50", "mAP"]
display_cols = [c for c in display_cols if c in df.columns]
print("\n" + df[display_cols].to_string(index=False))

csv_path = "/kaggle/working/magnitude_pruning_report.csv"
df.to_csv(csv_path, index=False)
print(f"\nReport saved to {csv_path}")

In [ ]:
# ===========================================================================
#  BENCHMARK SUMMARY: Before vs After Pruning (per ratio)
# ===========================================================================
print("\n\n========== BENCHMARK: BEFORE vs AFTER PRUNING ==========\n")
for model_name in ["DETR", "YOLOv5"]:
    sub = df[df["model"] == model_name]
    baseline = sub[sub["stage"] == "baseline"]
    print(f"\n--- {model_name} ---")
    hdr = f"{'Ratio':<8} {'Stage':<18} {'Params':<12} {'Active':<12} {'Sparsity':<10} "
    hdr += f"{'Compress':<10} {'SizeMB':<8} {'SparseMB':<10} {'FLOPs(G)':<10} "
    hdr += f"{'Lat(ms)':<10} {'FPS':<8} {'mAP50':<8} {'mAP':<8}"
    print(hdr)
    print("-" * 130)

    b = baseline.iloc[0]
    print(f"{'0.0':<8} {'baseline':<18} {b['params']:<12} {b['active_params']:<12} "
          f"{b['sparsity']*100:<9.1f}% {b['compression_ratio']:<10.2f} "
          f"{b['model_size_mb']:<8.1f} {b['sparse_size_mb']:<10.2f} "
          f"{b['flops_g']:<10.2f} {b['latency_ms']:<10.2f} {b['fps']:<8.1f} "
          f"{b['mAP50']:<8.3f} {b['mAP']:<8.3f}")

    for ratio in PRUNING_RATIOS[1:]:
        pruned = sub[(sub["stage"] == "after_pruning") & (sub["pruning_ratio"] == ratio)]
        if pruned.empty: continue
        p = pruned.iloc[0]
        print(f"{ratio:<8} {'after_pruning':<18} {p['params']:<12} {p['active_params']:<12} "
              f"{p['sparsity']*100:<9.1f}% {p['compression_ratio']:<10.2f} "
              f"{p['model_size_mb']:<8.1f} {p['sparse_size_mb']:<10.2f} "
              f"{p['flops_g']:<10.2f} {p['latency_ms']:<10.2f} {p['fps']:<8.1f} "
              f"{p['mAP50']:<8.3f} {p['mAP']:<8.3f}")

        recovered = sub[(sub["stage"] == "after_recovery") & (sub["pruning_ratio"] == ratio)]
        if not recovered.empty:
            r = recovered.iloc[0]
            print(f"{ratio:<8} {'after_recovery':<18} {r['params']:<12} {r['active_params']:<12} "
                  f"{r['sparsity']*100:<9.1f}% {r['compression_ratio']:<10.2f} "
                  f"{r['model_size_mb']:<8.1f} {r['sparse_size_mb']:<10.2f} "
                  f"{r['flops_g']:<10.2f} {r['latency_ms']:<10.2f} {r['fps']:<8.1f} "
                  f"{r['mAP50']:<8.3f} {r['mAP']:<8.3f}")
        print()

In [ ]:
# ===========================================================================
#  VISUALISATION — 6 subplots: mAP, mAP50, FPS, Latency, Sparse Size, Compression
# ===========================================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Magnitude Pruning: DETR-ResNet50 vs YOLOv5s | COCO val2017", fontsize=13)

COLORS  = {"DETR": "#2196F3", "YOLOv5": "#FF5722"}
METRICS = [
    ("mAP",             "mAP (IoU 0.50:0.95)",  axes[0, 0]),
    ("mAP50",           "mAP50",                 axes[0, 1]),
    ("fps",             "FPS",                   axes[0, 2]),
    ("latency_ms",      "Latency (ms/batch)",    axes[1, 0]),
    ("sparse_size_mb",  "Sparse size (MB)",      axes[1, 1]),
    ("compression_ratio","Compression ratio",    axes[1, 2]),
]

for metric, title, ax in METRICS:
    if metric not in df.columns:
        ax.set_visible(False); continue
    for mname in ["DETR", "YOLOv5"]:
        sub_b = df[(df["model"] == mname) & (df["stage"] == "baseline")]
        sub_p = df[(df["model"] == mname) & (df["stage"] == "after_pruning")]
        sub_r = df[(df["model"] == mname) & (df["stage"] == "after_recovery")]

        xs_p = [0.0] + list(sub_p["pruning_ratio"])
        ys_p = list(sub_b[metric]) + list(sub_p[metric])
        ax.plot(xs_p, ys_p, marker='o', label=f"{mname} pruned", color=COLORS[mname])

        if not sub_r.empty:
            xs_r = [0.0] + list(sub_r["pruning_ratio"])
            ys_r = list(sub_b[metric]) + list(sub_r[metric])
            ax.plot(xs_r, ys_r, marker='s', linestyle='--',
                    label=f"{mname} recovered", color=COLORS[mname], alpha=0.6)

    ax.set_title(title); ax.set_xlabel("Pruning ratio"); ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = "/kaggle/working/magnitude_pruning_plots.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {plot_path}")

In [ ]:
print("\nAll done. Output files:")
print(f"  {csv_path}")
print(f"  {plot_path}")